**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/peyrache-2015"], check=True)
    os.chdir("/content/encore/entries/peyrache-2015")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Peyrache et al. (2015) from DANDI:000056

**Paper.** Peyrache A, Lacroix MM, Petersen PC, Buzsáki G (2015). Internally organized mechanisms of the head direction sense. *Nature Neuroscience* 18, 569-575. doi:10.1038/nn.3968.

**Dataset.** DANDI:000056, version 0.261106.1949 (doi:10.48324/dandi.000056/0.261106.1949), streamed from the archive. The paper cites no version of the dandiset. This is the most recent published version, and the only one whose units carry a brain region (ADn or post-subiculum), the paper's sleep scoring and the open-field epoch. The earlier versions (0.210812.1518, 0.250624.0430) hold 40 sessions without regions, so they cannot separate the two structures that every claim compares; nothing here was read from them.

**Text.** The claims file `claims/peyrache-2015.yaml` was written from the NIH author manuscript (PMC4376557, Europe PMC) with its supplements (Supplementary Figures 1 to 8, Supplementary Table 1, reporting checklist). Every method below is from that file's criteria, or from a passage of the paper cited in `hd.py` or in the cell.

**Claims.** 35 claims. 32 are tested here, and two more (fig5d-adn-predicts-pos-reliably, fig5e-adn-predicts-pos, the GLM of Fig. 5d,e) in `glm_peyrache_2015.ipynb`, which runs on its own, as the GLM is slow (its run time is printed there). The last, fig5e-readout-windows, runs with its dense window grid in `glm_windows_peyrache_2015.ipynb`. It was first recorded as blocked (`too_large`); the third verification round agreed with the outcome that notebook gives. The table at the end gives every outcome.

**Units of both structures.** The files label units of four mice ADn and post-subiculum (PoS): Mouse24, 25 and 28, which the paper lists, and Mouse20, in 3 sessions, on electrode groups the lab labelled 'postsub' on a device the files call the post-subicular probe (data check). Every analysis follows the files' labels; the paper's three mice are shown as a variant where the choice matters.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time, resource
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", hd.DANDISET, "version", hd.VERSION)

In [ ]:
# Stream and analyse all 42 sessions (four at a time). The per-session
# measures are in hd.analyze_session; everything below aggregates them.
t0 = time.time()
RES = hd.run_all(workers=4)
print(f"analysed {len(RES)} sessions in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
SUM = pd.DataFrame([o["summary"] for o in RES])
CELLS = pd.DataFrame([c for o in RES for c in o["cells"]])
PAIRS = pd.DataFrame([c for o in RES for c in o["pairs"]])
# Ensembles of HD cells under (hd-a), with every measure, and under (hd-b),
# waking decoding only (decoding-tracks-hd, fig4d,e).
ENS_ALL = pd.DataFrame([{k: v for k, v in c.items() if k != "coherence"} for o in RES for c in o["ens"]])
ENS = ENS_ALL[ENS_ALL.reading == "hd-a"].reset_index(drop=True)
COH = pd.DataFrame([dict(session=c["session"], region=c["region"], state=k[0], ref=k[1], dist=k[2], mean=v[0], median=v[1])
                    for o in RES for c in o["ens"] if c["reading"] == "hd-a" for k, v in c["coherence"].items()])
FAST = pd.DataFrame([c for o in RES for c in o["fast"]])
MONO = pd.DataFrame([c for o in RES for c in o["mono"]])
DEC5B = pd.DataFrame([c for o in RES for c in o["dec5b"]])
LFP = pd.DataFrame([c for o in RES for c in o["lfp"]])
SHIFT = pd.DataFrame([dict(session=o["summary"]["session"], region=k[0], shift_s=k[1], err=v)
                      for o in RES for k, v in o["shift"].items()])
# Mice with both structures. The files label units of four mice ADn and PoS
# (Mouse20 too, on a device the files call the post-subicular probe; see the
# data check). The analyses follow the files' labels; the paper's list of
# three mice (Supplementary Table 1) is shown as a variant where it matters.
BOTH = sorted(SUM[(SUM.n_adn > 0) & (SUM.n_pos > 0)].subject.unique())
DUAL = ["Mouse24", "Mouse25", "Mouse28"]  # Supplementary Table 1: ADn and PoS recorded together
print("mice with units labelled ADn and PoS in one session:", BOTH, "| the paper's:", DUAL)
print({k: len(v) for k, v in dict(cells=CELLS, pairs=PAIRS, ens=ENS, ens_hd_b=ENS_ALL[ENS_ALL.reading == "hd-b"], fast=FAST, mono=MONO, dec5b=DEC5B, lfp=LFP).items()})

## Data check

What was loaded, per session, and each check of the description against the values.

In [ ]:
cols = ["session", "subject", "n_units", "n_adn", "n_pos", "other_regions", "has_hd", "has_states", "open_field",
        "sleep_pre", "sleep_post", "lfp", "wake_s", "SWS_s", "REM_s", "hd_valid_wake", "x_range_cm", "y_range_cm"]
print(SUM[cols].to_string())
print("\nsessions", len(SUM), "| mice", SUM.subject.nunique(), "| units", SUM.n_units.sum(),
      "| ADn units", SUM.n_adn.sum(), "| PoS units", SUM.n_pos.sum())
print("units per mouse and region:")
print(SUM.groupby("subject")[["n_units", "n_adn", "n_pos"]].sum().to_string())
print("\nsessions without head direction:", list(SUM.session[~SUM.has_hd]))
print("sessions without the paper's sleep scoring:", list(SUM.session[~SUM.has_states]))
print("sessions without LFP:", list(SUM.session[~SUM.lfp]))
print("spike times sorted in every unit of every session:", bool(SUM.spikes_sorted.all()))
print("first spike >= 0 s:", bool((SUM.spike_min >= 0).all()),
      "| sessions with spikes after the last sub-recording:",
      list(SUM.session[SUM.spike_max > SUM.rec_end + 1]), (SUM.spike_max - SUM.rec_end)[SUM.spike_max > SUM.rec_end + 1].round(1).tolist())
h = SUM[SUM.has_hd]
print(f"head direction: {h.hd_rate.unique()} Hz, range {h.hd_min.min():.4f} to {h.hd_max.max():.4f} rad; "
      f"valid (both LEDs) in wake {h.hd_valid_wake.min():.3f} to {h.hd_valid_wake.max():.3f}")
print("head direction = LED (red minus blue) angle minus a fixed offset: resultant length of the difference",
      f"{h.led_hd_R.min():.5f} to {h.led_hd_R.max():.5f}; offsets (deg) by mouse:")
print(h.groupby("subject").led_minus_hd_deg.agg(lambda x: sorted(set(np.round(x % 360, 0)))).to_string())
print("head-direction samples minus position samples:", h.hd_len_minus_pos_len.value_counts().to_dict())
print(f"wake extent of the red LED: x {h.x_range_cm.min():.1f} to {h.x_range_cm.max():.1f} cm, "
      f"y {h.y_range_cm.min():.1f} to {h.y_range_cm.max():.1f} cm (arena 53 x 46 cm, paper)")
print("wake (open field) min:", (h.wake_s / 60).round(1).describe()[["min", "50%", "max"]].to_dict())
print("SWS and REM periods inside the home-cage epochs (fraction):",
      SUM.SWS_in_homecage.min(), SUM.REM_in_homecage.min())
ov = SUM[SUM.wake_in_open_field.notna()]
print(f"scored wake interval against the open-field epochs, {len(ov)} sessions with both: fraction of wake inside "
      f"the open field {ov.wake_in_open_field.min():.4f} to {ov.wake_in_open_field.max():.4f}; fraction of the open field "
      f"inside wake {ov.open_field_in_wake.min():.4f} to {ov.open_field_in_wake.max():.4f}")
print("sessions with an open field but no scored wake (open field used as wake):",
      list(SUM.session[SUM.open_field & SUM.wake_in_open_field.isna()]))
print("\nsessions with a device named postsubicular_probe, by mouse:", SUM.groupby("subject").has_pos_probe.sum().to_dict())
print("electrode groups of the units labelled PoS in Mouse20:")
for _, r in SUM[(SUM.subject == "Mouse20") & (SUM.n_pos > 0)].iterrows():
    print(" ", r.session, r.n_pos, "units |", r.pos_groups)

In [ ]:
# Clock check (protocol, data check): waking decoding error with the head
# direction shifted in time. If spikes and head direction share a clock, the
# error is smallest at 0 s.
piv = SHIFT.pivot_table(index=["session", "region"], columns="shift_s", values="err")
print(piv.round(1).to_string())
best = piv.idxmin(axis=1)
print("\nshift with the smallest error, count of ensembles:", best.value_counts().to_dict())
print("median error over ensembles by shift (deg):", piv.median().round(1).to_dict())

In [ ]:
# Per-cell table: what the analysis uses.
print(CELLS.groupby(["subject", "region"]).size().unstack().to_string())
for k in ["rate_wake", "rate_SWS", "rate_REM", "peak_a", "kappa_a", "info", "dur_ms", "t2p_ms"]:
    v = CELLS[k].replace([np.inf, -np.inf], np.nan)
    print(f"{k}: n {v.notna().sum()}, range {v.min():.3g} to {v.max():.3g}, median {v.median():.3g}")
print("classifiable cells (sessions with head direction):", len(CELLS))
print("agreement of HD reading (hd-a) with the file's is_hd_cell column:")
print(pd.crosstab(CELLS.hd_a, CELLS.lab_hd, rownames=["hd_a"], colnames=["file is_hd_cell"]))
print(pd.crosstab(CELLS.hd_b, CELLS.lab_hd, rownames=["hd_b"], colnames=["file is_hd_cell"]))
print("PoS cell classes from the waveform:", CELLS[CELLS.region == "PoS"].cls.value_counts().to_dict())

### The session rule of the Methods

"To prevent statistical bias of neuron sampling, we discarded from analysis sessions separated by less than three days during which the thalamic probe was not moved" (Methods, Electrodes). The files give the thalamic probe depth of each session (device description, from the CRCNS table). `hd.thalamic_rule` keeps a session when the depth changed since the last kept session or three days have passed. Supplementary Table 1 counts ADn cells from 2, 3 and 3 sessions in Mouse24, 25 and 28, and PoS cells from all of their 4, 10 and 7. The rule is shown as a variant of the cell counts; the main analysis uses every session, as the files do not say which sessions the paper kept.

In [ ]:
import re as _re
deps = []
for o in RES:
    deps.append((o["summary"]["session"], o["summary"].get("thal_depth")))
rule = hd.thalamic_rule(deps)
SUM["thal_depth"] = [d for _, d in deps]
SUM["rule_keep"] = SUM.session.map(rule)
print(SUM[["session", "thal_depth", "rule_keep"]].to_string())
print(SUM.groupby("subject").rule_keep.sum().to_dict())
CELLS["rule_keep"] = CELLS.session.map(rule)

## recording-sample

> HD cell ensembles were recorded in 7 mice over 42 open-field sessions with home-cage sleep, with ADn and PoS recorded simultaneously in 3 mice over 21 sessions.

Readings (criterion): (a) all sessions, (b) sessions with sleep before and after the open field.

In [ ]:
s_open = SUM[SUM.open_field & (SUM.n_units > 0)]
s_sleep = s_open[s_open.sleep_pre & s_open.sleep_post]
print("sessions in the dandiset:", len(SUM), "| mice:", SUM.subject.nunique())
print("without an open-field epoch:", list(SUM.session[~SUM.open_field]))
rows = []
for rd, sel in (("(a) open field and spikes", s_open), ("(b) and sleep before and after", s_sleep)):
    both = sel[(sel.n_adn > 0) & (sel.n_pos > 0)]
    both_p = sel[sel.subject.isin(DUAL)]
    rows.append(dict(reading=rd, mice=sel.subject.nunique(), sessions=len(sel),
                     both_sessions_labels=len(both), both_mice_labels=both.subject.nunique(),
                     both_sessions_paper_list=len(both_p), both_mice_paper_list=both_p.subject.nunique()))
print(pd.DataFrame(rows).to_string(index=False))
print("expected: 7 mice (exact), 42 sessions (+-4.2), 21 sessions with both (+-2.1), 3 mice with both (exact)")
print("sessions per mouse:", SUM.groupby("subject").size().to_dict())
both_lab = s_open[(s_open.n_adn > 0) & (s_open.n_pos > 0)]
print("sessions with units of both labels, by mouse (a):", both_lab.groupby("subject").size().to_dict())
print("sessions without sleep before and after:", list(s_open.session[~(s_open.sleep_pre & s_open.sleep_post)]))

**Result: partially reproduced.** Under reading (a), the dandiset holds 7 mice and 41 sessions with an open-field epoch and spikes (Mouse32-140820 has no epochs), both within tolerance. By the files' labels, 24 of these sessions in 4 mice hold units of both structures, against 21 sessions (tolerance 2.1) in 3 mice: Mouse20 has PoS units in 3 sessions. Under reading (b) (40 sessions with sleep before and after) the counts are 23 sessions in 4 mice; the session count is then within tolerance, the mouse count is not. Only with the paper's own list of three mice (a variant from Supplementary Table 1, not from the data) are the counts 21 and 3. The table's 50 sessions are not matched.

## Head-direction cells: supptable1-per-animal, hd-cell-fraction, hd-fraction-hd-shanks, ensemble-sizes

HD cells as in the claims file's opening comment: tuning curve in 1 degree bins smoothed with a 6 degree s.d. Gaussian in waking (the open-field epoch), frames with either LED missing excluded; von Mises concentration > 1, peak rate > 1 Hz, Rayleigh p < 0.001; readings (hd-a) and (hd-b) (`hd.vm_fit_a`, `hd.vm_fit_b`). Cells of sessions without head direction (Mouse28-140314, both Mouse32 sessions) cannot be classified.

In [ ]:
EXP_ADN = dict(Mouse12=63, Mouse17=52, Mouse20=15, Mouse24=17, Mouse25=23, Mouse28=37, Mouse32=35)
EXP_POS = dict(Mouse24=9, Mouse25=53, Mouse28=49)
REC_ADN = dict(Mouse12=217, Mouse17=245, Mouse20=49, Mouse24=22, Mouse25=27, Mouse28=55, Mouse32=105)
REC_POS = dict(Mouse24=48, Mouse25=135, Mouse28=174)
rec_units = SUM.groupby("subject")[["n_adn", "n_pos"]].sum()
rec_rule = SUM[SUM.rule_keep.fillna(True).astype(bool)].groupby("subject")[["n_adn"]].sum()
testable = set(SUM[SUM.has_hd].subject)
def table(sel, label):
    rows = []
    for reg, exp in (("ADn", EXP_ADN), ("PoS", EXP_POS)):
        for m, v in exp.items():
            na = int(sel[(sel.subject == m) & (sel.region == reg)].hd_a.sum())
            nb = int(sel[(sel.subject == m) & (sel.region == reg)].hd_b.sum())
            nl = int(sel[(sel.subject == m) & (sel.region == reg)].lab_hd.fillna(False).astype(bool).sum())
            rows.append(dict(region=reg, mouse=m, expected=v, testable=m in testable, hd_a=na, hd_b=nb, file_is_hd=nl,
                             within_a=abs(na - v) <= 0.1 * v, within_b=abs(nb - v) <= 0.1 * v))
    T = pd.DataFrame(rows)
    print(label)
    print(T.to_string(index=False))
    print("within tolerance: hd-a", int(T.within_a.sum()), "of 10; hd-b", int(T.within_b.sum()), "of 10;",
          int(T.testable.sum()), "counts testable (no head direction in any session of",
          sorted(set(EXP_ADN) - testable), ")")
    return T
T_all = table(CELLS, "all sessions")
T_rule = table(CELLS[(CELLS.region == "PoS") | CELLS.rule_keep.astype(bool)], "\nADn cells from sessions kept by the session rule (variant)")
print("\nrecorded units per mouse (dandiset, all sessions) against Supplementary Table 1:")
print(pd.DataFrame(dict(ADn=rec_units.n_adn, ADn_table=pd.Series(REC_ADN), ADn_rule=rec_rule.n_adn,
                        PoS=rec_units.n_pos, PoS_table=pd.Series(REC_POS))).to_string())

**Result: partially reproduced.** Under reading (hd-b), 5 of the 10 HD counts are within tolerance, and under (hd-a) 3. The session-rule variant gives 5 under (hd-b) and 4 under (hd-a). Mouse32's count (35) cannot be tested: neither of its sessions holds head direction, so 9 of the 10 counts are testable and its 0 is not a measurement. The outcome would be the same with Mouse32 left out: 5 of the 9 testable counts are within tolerance under (hd-b), and a passing Mouse32 would make 6. Over all sessions the dual mice have more ADn HD cells (26, 62 and 74 under hd-a, against 17, 23 and 37), as Supplementary Table 1 counts their ADn cells from a subset of sessions. Recorded units equal the table in Mouse12, Mouse17 and Mouse32 (217, 245, 105), Mouse20 has 48 against 49, and the PoS counts match (48, 135, 174).

In [ ]:
def fractions(sel, rd):
    out = {}
    for reg, key in (("ADn", "thal"), ("PoS", "pos")):
        c = sel[sel.region == reg]
        out[key + "_hd"] = int(c["hd_" + rd].sum())
        out[key + "_rec"] = len(c)
        ps = c.groupby("session")["hd_" + rd].mean()
        pm = c.groupby("subject")["hd_" + rd].mean()
        out[key + "_frac_sessions"] = ps.mean()
        out[key + "_frac_mice"] = pm.mean()
        out[key + "_frac_pooled"] = c["hd_" + rd].mean()
    return out
rows = []
for lab, sel in (("all sessions", CELLS), ("session rule", CELLS[(CELLS.region == "PoS") | CELLS.rule_keep.astype(bool)])):
    for rd in ("a", "b"):
        r = fractions(sel, rd); r.update(sample=lab, reading=rd); rows.append(r)
F = pd.DataFrame(rows)
print(F.round(3).to_string(index=False))
print("expected: thalamic 242 of 720, 41% on average; PoS 111 of 357, 31%")
print("recorded ADn units including sessions without head direction:", int(SUM.n_adn.sum()),
      "| PoS:", int(SUM.n_pos.sum()), "| of these classifiable:", int((CELLS.region == "ADn").sum()), int((CELLS.region == "PoS").sum()))
print("recorded units in the dual mice under the session rule (ADn):", int(SUM[SUM.rule_keep.fillna(True).astype(bool)].n_adn.sum()))

**Result: not reproduced.** No combination of readings puts all four counts within tolerance. Over all sessions, reading (hd-a) gives 293 thalamic HD cells of 727 classifiable thalamic cells, and 82 PoS HD cells of 337. Reading (hd-b) gives 312 and 120. The per-session mean fractions are 0.53 and 0.22 (hd-a), and 0.57 and 0.33 (hd-b). The PoS counts (337 classifiable, 365 units), the hd-b PoS HD count (120) and the hd-b per-session PoS fraction (0.33) are within tolerance. The thalamic HD counts exceed 242 because the dual mice add sessions the paper did not count; under the session rule they are 207 and 220 of 593.

In [ ]:
rows = []
for rd in ("a", "b"):
    a = CELLS[CELLS.region == "ADn"].groupby(["session", "shank"])["hd_" + rd].agg(["sum", "count"])
    a = a[a["sum"] > 0]
    rows.append(dict(reading=rd, shank_sessions=len(a), mean_of_shanks=(a["sum"] / a["count"]).mean(),
                     pooled=a["sum"].sum() / a["count"].sum()))
print(pd.DataFrame(rows).round(3).to_string(index=False), "\nexpected 0.71 (tolerance 0.071)")

**Result: reproduced.** On 63 thalamic shank-sessions with at least one HD cell, the mean HD fraction is 0.733 under (hd-a) and the pooled fraction 0.736, both within 0.639 to 0.781. Under (hd-b) they are 0.799 and 0.784, outside the tolerance.

In [ ]:
rows = []
for rd in ("a", "b"):
    for m20 in ("with Mouse20", "without Mouse20"):
        for reg in ("ADn", "PoS"):
            c = CELLS[CELLS.region == reg]
            if m20 == "without Mouse20":
                if reg == "ADn":
                    continue  # Mouse20's ADn cells are in the paper's sample either way
                c = c[c.subject != "Mouse20"]
            per = c.groupby("session")["hd_" + rd].sum()
            for lab, p in (("(a) sessions with the structure", per), ("(b) sessions with >= 1 HD cell", per[per > 0]),
                           ("(c) sessions with >= 6 HD cells (decoding rule)", per[per >= 6])):
                mean, sd = p.mean(), p.std(ddof=1)
                exp_m, exp_s = (8.4, 5.1) if reg == "ADn" else (5.0, 2.8)
                rows.append(dict(hd=rd, pos_sample=m20 if reg == "PoS" else "", region=reg, reading=lab, n=len(p),
                                 mean=mean, sd=sd, mean_ok=abs(mean - exp_m) <= 0.1 * exp_m, sd_ok=abs(sd - exp_s) <= 0.1 * exp_s))
ES = pd.DataFrame(rows)
print(ES.round(2).to_string(index=False))
print("expected ADn 8.4 +- 5.1 s.d., PoS 5 +- 2.8 s.d.; tolerance 10% of each value")
best = []
for rd in ("a", "b"):
    for smp in ("with Mouse20", "without Mouse20"):
        for lab in ES.reading.unique():
            a = ES[(ES.hd == rd) & (ES.region == "ADn") & (ES.reading == lab)].iloc[0]
            p = ES[(ES.hd == rd) & (ES.region == "PoS") & (ES.pos_sample == smp) & (ES.reading == lab)].iloc[0]
            best.append(dict(hd=rd, pos_sample=smp, reading=lab, both_means=a.mean_ok & p.mean_ok,
                             all_four=a.mean_ok & p.mean_ok & a.sd_ok & p.sd_ok))
print(pd.DataFrame(best).to_string(index=False))
print("by mouse (ADn, hd-a / hd-b):", CELLS[CELLS.region == "ADn"].groupby(["subject", "session"])[["hd_a", "hd_b"]].sum()
      .groupby("subject").mean().round(1).to_dict())

**Result: partially reproduced.** Under reading (hd-b) with reading (a), both means are within tolerance: ADn HD cells per session are 8.00 over 39 sessions (7.56 to 9.24) and PoS HD cells 5.22 over 23 sessions (4.5 to 5.5). The s.d. are not: 4.42 against 5.1 plus or minus 0.51, and 4.11 against 2.8 plus or minus 0.28. No other combination puts both means within tolerance. The PoS mean rests on the 3 Mouse20 sessions with PoS units (no HD cells among them): without them it is 6.00 over 20 sessions, outside. Under (hd-a), the means are 7.51 and 3.57 (a), 7.51 and 4.10 (b), 10.04 and 8.20 (c).

In [ ]:
sl = SUM[SUM.has_states & SUM.sleep_pre | SUM.has_states & SUM.sleep_post].copy()
sl["homecage_h"] = sl.homecage_s / 3600
sl["scored_h"] = (sl.SWS_s + sl.REM_s) / 3600
print(sl[["session", "homecage_h", "scored_h"]].round(2).to_string(index=False))
for k in ("homecage_h", "scored_h"):
    print(f"{k}: n {len(sl)}, mean {sl[k].mean():.2f} h, s.d. {sl[k].std(ddof=1):.2f} h")
print("expected 5 +- 1 s.d. h (tolerance 0.5 and 0.1)")
print("by mouse (home cage, h):", sl.groupby("subject").homecage_h.mean().round(2).to_dict())

**Result: reproduced.** Under reading (a), the home-cage epochs before and after the open field total 4.99 h on average, with an s.d. of 0.97 h, over 37 sessions with the paper's scoring. Both values are within tolerance. The scored SWS and REM total 3.20 plus or minus 0.57 h (reading b), outside it.

## Firing and information: adn-pos-peak-rate, adn-pos-hd-information, supfig2a-unimodality

In [ ]:
rows = []
for rd in ("a", "b"):
    h = CELLS[CELLS["hd_" + rd]]
    A, P = h[h.region == "ADn"]["peak_" + rd], h[h.region == "PoS"]["peak_" + rd]
    u = stats.mannwhitneyu(A, P, alternative="two-sided")
    rows.append(dict(reading=rd, n_adn=len(A), n_pos=len(P), median_adn=A.median(), median_pos=P.median(),
                     ratio_medians=A.median() / P.median(), mean_adn=A.mean(), mean_pos=P.mean(),
                     ratio_means=A.mean() / P.mean(), U=u.statistic, p=u.pvalue))
print(pd.DataFrame(rows).to_string(index=False))
print("expected ratio 3 (tolerance 0.3, or 2.5 to 3), Mann-Whitney p < 1e-10")
h = CELLS[CELLS.hd_a]
print("by mouse, median peak rate (hd-a):")
print(h.groupby(["subject", "region"]).peak_a.median().unstack().round(2).to_string())

**Result: partially reproduced.** ADn HD cells have higher peak rates than PoS HD cells (Mann-Whitney p = 1.9e-13 under hd-a, 1.9e-18 under hd-b, both below 1e-10). The ratio is above 3.3 under every reading: medians 3.76 and means 3.95 (hd-a), medians 3.51 and means 4.26 (hd-b).

In [ ]:
rows = []
for rd in ("a", "b"):
    h = CELLS[CELLS["hd_" + rd]]
    A, P = h[h.region == "ADn"]["info"], h[h.region == "PoS"]["info"]
    u = stats.mannwhitneyu(A, P, alternative="two-sided")
    rows.append(dict(reading=rd, n_adn=len(A), n_pos=len(P), median_adn=A.median(), median_pos=P.median(),
                     mean_adn=A.mean(), mean_pos=P.mean(), U=u.statistic, p=u.pvalue))
print(pd.DataFrame(rows).to_string(index=False))
print("expected ADn 1.44, PoS 1.11 bit/spike (tolerance 10%), Mann-Whitney p < 1e-6")
print("by mouse, median (hd-a):")
print(h.groupby(["subject", "region"])["info"].median().unstack().round(2).to_string())

**Result: reproduced.** Under reading (hd-b) with means, ADn HD cells carry 1.38 and PoS HD cells 1.00 bits per spike, both within tolerance, with Mann-Whitney p = 3.9e-11. Under (hd-a) both medians (1.38, 1.14) and both means (1.44, 1.17) are within tolerance, but p = 4.5e-5 does not reach 1e-6. With (hd-b) medians the PoS value (0.87) is outside.

In [ ]:
rows = []
for rd in ("a", "b"):
    h = CELLS[CELLS["hd_" + rd]]
    for reg in ("ADn", "PoS"):
        c = h[h.region == reg]
        rows.append(dict(hd_reading=rd, region=reg, n=len(c), not_unimodal_a=c.unimodal_fail_a.mean(),
                         not_unimodal_b=c.unimodal_fail_b.mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("expected ADn 0.12, PoS 0.32 (tolerance 10%)")

**Result: reproduced.** Under HD reading (hd-b) and peak reading (a), 12.2% of ADn and 29.2% of PoS HD cells have a second peak of 50% or more, both within tolerance. Under (hd-a) the ADn fraction (10.9%) is within and the PoS fraction (18.3%) is not. Peak reading (b) gives lower fractions (3.4 to 5.1% ADn, 12.2 to 24.2% PoS).

## State preservation: fig2a-rates-preserved, fig2b-pair-correlations-preserved

In [ ]:
rows = []
for rd in ("a", "b"):
    for sel_name in ("sessions with SWS and REM", "rate-a", "rate-b"):
        h = CELLS[CELLS["hd_" + rd] & CELLS.has_sleep]
        if sel_name == "rate-a":
            h = h[h.pass_rate_a]
        if sel_name == "rate-b":
            h = h[h.pass_rate_b]
        for reg in ("ADn", "PoS"):
            c = h[h.region == reg]
            for st in ("SWS", "REM"):
                r, p = stats.pearsonr(c.rate_wake, c["rate_" + st])
                rl, _ = stats.pearsonr(np.log10(c.rate_wake + 1e-3), np.log10(c["rate_" + st] + 1e-3))
                rows.append(dict(hd=rd, cells=sel_name, region=reg, state=st, n=len(c), r=r, p=p, r_log=rl))
F2A = pd.DataFrame(rows)
print(F2A.round(4).to_string(index=False))
print("expected n: ADn 215, PoS 62; threshold readings r >= 0.7 or r >= 0.5")
h = CELLS[CELLS.hd_a & CELLS.has_sleep]
print("by mouse (hd-a, ADn and PoS together), r wake-SWS / wake-REM:")
for m, c in h.groupby("subject"):
    if len(c) > 3:
        print(m, len(c), round(stats.pearsonr(c.rate_wake, c.rate_SWS)[0], 3), round(stats.pearsonr(c.rate_wake, c.rate_REM)[0], 3))

**Result: reproduced.** Under the r of at least 0.5 reading, all four correlations pass with p < 0.05. In ADn the waking rates correlate with SWS rates at r = 0.80 and with REM rates at r = 0.86 (hd-a, 274 cells in sessions with both sleep stages). In PoS the values are 0.65 and 0.66 (69 cells). The PoS values are below 0.7 under every cell reading except waking against SWS with rate-a cells (0.73). The cell counts are 245 to 293 ADn and 38 to 100 PoS, against 215 and 62.

In [ ]:
# Pairs of HD cells with their offsets and the rate threshold flags.
C_idx = CELLS.set_index(["session", "unit"])
def attach(df, a="u1", b="u2"):
    df = df.copy()
    for side, col in (("1", a), ("2", b)):
        key = list(zip(df.session, df[col]))
        sub = C_idx.loc[key]
        for k in ("region", "hd_a", "hd_b", "pref_a", "pref_b", "pass_rate_a", "pass_rate_b", "rate_wake", "rate_SWS",
                  "rate_REM", "peak_a", "has_sleep"):
            df[k + side] = sub[k].values
    df["off_a"] = np.degrees(np.abs(hd.circ_diff(df.pref_a1.values.astype(float), df.pref_a2.values.astype(float))))
    df["off_b"] = np.degrees(np.abs(hd.circ_diff(df.pref_b1.values.astype(float), df.pref_b2.values.astype(float))))
    return df
PP = attach(PAIRS)
def hdpairs(rd, rate):
    m = PP["hd_" + rd + "1"].astype(bool) & PP["hd_" + rd + "2"].astype(bool)
    m &= PP["pass_rate_" + rate + "1"].astype(bool) & PP["pass_rate_" + rate + "2"].astype(bool)
    return PP[m]
print("all pairs of ADn and PoS cells:", len(PP), PP.groupby("kind").size().to_dict())
for rd in ("a", "b"):
    for rate in ("a", "b"):
        print(f"hd-{rd}, rate-{rate}:", hdpairs(rd, rate).groupby("kind").size().to_dict())

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        for ss in ("all pairs", "different shanks"):
            P_ = hdpairs(rd, rate)
            P_ = P_[P_.has_sleep1.astype(bool)]
            if ss == "different shanks":
                P_ = P_[~P_.same_shank]
            for kind, c in P_.groupby("kind"):
                for st in ("SWS", "REM"):
                    ok = c[["corr_wake", "corr_" + st]].notna().all(1)
                    r, p = stats.pearsonr(c.corr_wake[ok], c["corr_" + st][ok])
                    rows.append(dict(hd=rd, rate=rate, pairs=ss, kind=kind, state=st, n=int(ok.sum()), r=r, p=p))
F2B = pd.DataFrame(rows)
print(F2B.round(4).to_string(index=False))
print("expected pairs: ADn 970 (Fig. 2b) or 907 (Fig. 3a), PoS 92")

**Result: reproduced.** The waking pair correlations predict the SWS and REM ones in every pair type, reading and shank selection, with r from 0.81 to 0.97 (all p < 0.05), so the 0.7 reading holds. The ADn pair counts (1006 to 1425 HD pairs) are above 970.

## Speed of the internal HD signal: fig2f-sws-tenfold-faster, fig2e-decoded-speed, fig2f-rem-similar-to-wake

The cross-correlogram speed fit is `hd.fit_speed` (Methods, Estimation of angular velocity): 10 ms bins, lags to +-2 s in wake and REM and +-0.5 s in SWS (reading v-a), chi-squared (3 d.f.) speed distribution, coarse grid then Nelder-Mead. Speeds are in degrees per second.

In [ ]:
def speed_pairs(rd, rate, st):
    """HD pairs of one type with a positive, finite fitted speed in wake and in state st."""
    P_ = hdpairs(rd, rate)
    P_ = P_[P_.kind.isin(["ADn-ADn", "PoS-PoS"])]
    ok = np.isfinite(P_.v_wake) & np.isfinite(P_["v_" + st]) & (P_.v_wake > 0) & (P_["v_" + st] > 0)
    return P_[ok], int((~ok).sum())
def at_edge(c, st):
    return (c.v_wake < 5.01) | (c.v_wake > 4999) | (c["v_" + st] < 5.01) | (c["v_" + st] > 4999)
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_, ndrop = speed_pairs(rd, rate, "SWS")
        for kind, c0 in P_.groupby("kind"):
            for edge in ("all fits", "no fit at a grid end"):
                c = c0 if edge == "all fits" else c0[~at_edge(c0, "SWS")]
                ratio = c.v_SWS / c.v_wake
                w = stats.wilcoxon(np.log(ratio))
                rows.append(dict(hd=rd, rate=rate, kind=kind, fits=edge, n=len(c), dropped_zero=ndrop, v_wake=c.v_wake.median(),
                                 v_SWS=c.v_SWS.median(), median_ratio=ratio.median(), frac_SWS_faster=(ratio > 1).mean(),
                                 wilcoxon_p=w.pvalue))
F2F = pd.DataFrame(rows)
print(F2F.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("expected median SWS/wake ratio 10 (tolerance 1)")
c = hdpairs("a", "a")
c = c[c.kind == "ADn-ADn"]
print("ADn-ADn (hd-a, rate-a) by mouse, median ratio:", (c.v_SWS / c.v_wake).groupby(c.subject).median().round(2).to_dict())
SP = PP[PP.kind.isin(["ADn-ADn", "PoS-PoS"]) & PP.v_wake.notna()]
print("fit quality per state, HD pairs of one structure (residual sum of squares over the CCG's sum of squares about its mean):")
for st in ("wake", "SWS", "REM"):
    v, r = SP["v_" + st], SP["res_" + st]
    print(f"  {st}: pairs {int(v.notna().sum())}; residual median {r.median():.3f}, 90th pct {r.quantile(0.9):.3f}, "
          f"fraction above 1 {float((r > 1).mean()):.3f}; speed at the grid ends (<= 5 or >= 5000 deg/s) {int(((v < 5.01) | (v > 4999)).sum())}; "
          f"speed 0 {int((v == 0).sum())}, below 1 deg/s {int((v < 1).sum())}; range {v.min():.3g} to {v.max():.4g} deg/s")
print("pairs with a fitted speed at a grid end in some state:",
      int(((SP[["v_wake", "v_SWS", "v_REM"]] < 5.01) | (SP[["v_wake", "v_SWS", "v_REM"]] > 4999)).any(axis=1).sum()), "of", len(SP))

**Result: partially reproduced.** The SWS speed exceeds the waking speed in most pairs (Wilcoxon p < 0.001 in every reading), but the median SWS to waking ratio is outside 9 to 11. In ADn-ADn pairs it is 11.65 to 12.31 with all fits and 11.04 to 11.71 without pairs whose fit sits at an end of the speed grid; 95.7% of ADn-ADn pairs are faster in SWS under hd-a and rate-a. In PoS-PoS pairs it is 5.18 to 6.15. It is above 5 in both, the partial criterion. No fitted speed is exactly 0, but every state has fits at 1.5e-14 deg/s, effectively 0 (the count below 1 deg/s is printed per state); they are among the fits at a grid end, so the variant without those fits leaves them out and gives the 11.04 to 11.71 above. Pairs without a finite speed in both states (107 or 156 under rate-b, sessions without sleep scoring) are left out. Fits are loose: the median residual is about half of the cross-correlogram's variance in waking and SWS and about equal to it in REM, and 387 of 1950 pairs have a speed at a grid end in some state.

In [ ]:
cols = ["session", "region", "n_hd", "speed_af_wake", "speed_b_wake", "speed_a_actual", "speed_b_actual",
        "speed_af_SWS", "speed_b_SWS", "speed_af_REM", "speed_b_REM"]
print(ENS[cols].round(1).to_string(index=False))
rows = []
E_ = ENS[ENS.speed_a_SWS.notna()]
STEPS = {"a, sub-degree": ("af", "a"), "a, 1-degree grid": ("a", "a"), "b": ("b", "b")}
for reg, e in E_.groupby("region"):
    for step, (key, akey) in STEPS.items():
        for wk in ("w-a decoded", "w-b actual"):
            for summ in ("median", "mean"):
                if summ == "mean" and key == "a":
                    continue
                k_ = {"af": "afmean", "b": "bmean"}.get(key, key) if summ == "mean" else key
                ka = {"a": "afmean", "b": "bmean"}[akey] if summ == "mean" else akey
                w = e[f"speed_{k_}_wake"] if wk.startswith("w-a") else e[f"speed_{ka}_actual"]
                sws, rem = e[f"speed_{k_}_SWS"].median(), e[f"speed_{k_}_REM"].median()
                rs = sws / w.median() if w.median() > 0 else np.inf
                rr = rem / w.median() if w.median() > 0 else np.inf
                rows.append(dict(region=reg, step=step, wake=wk, per_ensemble=summ, n=len(e), wake_med=w.median(), SWS_med=sws,
                                 REM_med=rem, ratio_SWS=rs, ratio_REM=rr, SWS_ok=9 <= rs <= 11, REM_ok=0.67 <= rr <= 1.5))
F2E = pd.DataFrame(rows)
print(F2E.round(3).to_string(index=False))
print("expected SWS/wake 10 (tolerance 1); REM/wake between 0.67 and 1.5. Per-ensemble value: the median speed (criterion),"
      " or the mean speed (variant). Columns *_med: median over ensembles.")
print("steps spanning two intervals of a state are left out; reading (a) at the 1-degree grid gives 0 or 100 deg/s medians")

**Result: partially reproduced.** The SWS part fails under every reading; the REM part holds in PoS only. With the per-ensemble median speed (the criterion), the ADn SWS to waking ratio is 5.11 at 10 ms steps with the posterior maximum located below the 1 degree grid (reading a), 6.00 at the smoothing window (b), and 1.22 or 4.33 with the actual head speed in waking (w-b); the ADn REM ratios are 0.15 to 0.63, below 0.67. In PoS (4 ensembles with sleep) the SWS ratios are 0.46 to 2.82 and the REM ratio is 0.93 (a, w-a) and 1.00 (b, w-a), inside 0.67 to 1.5. On the 1 degree grid, 10 ms medians are 0 or 100 deg/s and cannot decide. As a variant outside the criterion, mean speeds per ensemble give ADn ratios of 10.26 (SWS) and 0.88 (REM) at the smoothing window with the actual waking speed.

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_, ndrop = speed_pairs(rd, rate, "REM")
        for kind, c in P_.groupby("kind"):
            ratio = c.v_REM / c.v_wake
            w = stats.wilcoxon(np.log(ratio))
            rows.append(dict(hd=rd, rate=rate, kind=kind, n=len(c), dropped_zero=ndrop, v_wake=c.v_wake.median(),
                             v_REM=c.v_REM.median(), median_ratio=ratio.median(), wilcoxon_p=w.pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("readings: (a) median ratio in 0.67 to 1.5; (b) Wilcoxon of the log ratio p >= 0.05")

**Result: reproduced.** Both readings hold in ADn-ADn and PoS-PoS pairs under every HD and rate reading. The median REM to waking speed ratio is 0.96 to 1.20, inside 0.67 to 1.5, and no Wilcoxon test of the log ratio reaches p < 0.05 (p from 0.17 to 0.66). Pairs without a positive, finite speed in both states are left out. No speed is exactly 0, but some fits sit at 1.5e-14 deg/s, effectively 0 (counts in the fig2f-sws-tenfold-faster cell); they are kept, and the medians do not rest on them.

## The activity packet: fig3a-packet-60-degrees, fig3a-adn-stronger-than-pos, supfig5-rate-independent, fig3b-packet-coherence

In [ ]:
def crossing(off, corr, width):
    b = np.arange(0, 180 + width, width)
    cen = 0.5 * (b[1:] + b[:-1])
    m = np.array([np.mean(corr[(off >= lo) & (off < hi)]) if np.any((off >= lo) & (off < hi)) else np.nan
                  for lo, hi in zip(b[:-1], b[1:])])
    for i in range(len(m) - 1):
        if m[i] > 0 and m[i + 1] <= 0:
            return cen[i] + (cen[i + 1] - cen[i]) * m[i] / (m[i] - m[i + 1])
    return np.nan
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_ = hdpairs(rd, rate)
        for kind, c in P_.groupby("kind"):
            for st in ("wake", "SWS", "REM"):
                ok = c["corr_" + st].notna()
                off = c["off_" + rd][ok].values
                cc = c["corr_" + st][ok].values
                ra, pa = stats.pearsonr(off, cc)
                rb, pb = stats.pearsonr(np.cos(np.radians(off)), cc)
                rows.append(dict(hd=rd, rate=rate, kind=kind, state=st, n=int(ok.sum()), r_abs=ra, p_abs=pa, r_cos=rb, p_cos=pb,
                                 cross10=crossing(off, cc, 10), cross20=crossing(off, cc, 20)))
F3A = pd.DataFrame(rows)
print(F3A.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("expected crossing 60 deg (tolerance 6); pairs ADn-ADn 907, ADn-PoS 512, PoS-PoS 92; Pearson p < 1e-6")
ok = F3A[(F3A.r_abs < 0) & (F3A.p_abs < 1e-6) & (F3A.r_cos > 0) & (F3A.p_cos < 1e-6)]
for rd in ("a", "b"):
    for rate in ("a", "b"):
        f = F3A[(F3A.hd == rd) & (F3A.rate == rate)]
        print(f"hd-{rd} rate-{rate}: Pearson (abs offset) falling with p < 1e-6 in {int(((f.r_abs < 0) & (f.p_abs < 1e-6)).sum())} of 9;"
              f" (cos) {int(((f.r_cos > 0) & (f.p_cos < 1e-6)).sum())} of 9; crossing within 54-66 deg: 10-deg bins"
              f" {int(f.cross10.between(54, 66).sum())}, 20-deg bins {int(f.cross20.between(54, 66).sum())} of 9")
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
P_ = hdpairs("a", "a")
for i, st in enumerate(("wake", "SWS", "REM")):
    for kind, c in P_.groupby("kind"):
        b = np.arange(0, 190, 10)
        m = [c["corr_" + st][(c.off_a >= lo) & (c.off_a < lo + 10)].mean() for lo in b[:-1]]
        ax[i].plot(b[:-1] + 5, m, label=kind)
    ax[i].axhline(0, color="k", lw=0.5); ax[i].set_title(st); ax[i].set_xlabel("offset (deg)")
ax[0].legend(); ax[0].set_ylabel("mean correlation"); plt.tight_layout(); plt.show()

**Result: partially reproduced.** Under hd-a and rate-b, the zero-lag correlation falls with the preferred-direction offset in all 9 state and pair-type combinations (p < 1e-6 with the absolute offset and with its cosine). The zero crossing is within 54 to 66 degrees in 7 of 9 combinations with 10 degree bins, and 8 of 9 with 20 degree bins. PoS-PoS in SWS crosses at 54 and 108 degrees. Under hd-a and rate-a the PoS-PoS tests reach only p = 5.5e-6 to 7.2e-5 with the absolute offset and 8.2e-6 to 1.3e-4 with its cosine (36 pairs); under hd-b and rate-a all 9 tests pass. ADn-ADn pairs cross at 54 to 59 degrees in every state.

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_ = hdpairs(rd, rate)
        P_ = P_[P_["off_" + rd] < 60]
        for st in ("wake", "SWS", "REM"):
            A = P_[P_.kind == "ADn-ADn"]["corr_" + st].dropna()
            B = P_[P_.kind == "PoS-PoS"]["corr_" + st].dropna()
            u = stats.mannwhitneyu(A, B, alternative="two-sided")
            rows.append(dict(hd=rd, rate=rate, state=st, n_adn=len(A), n_pos=len(B), med_adn=A.median(), med_pos=B.median(), p=u.pvalue))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("expected pairs below 60 deg: ADn-ADn 345, PoS-PoS 55")

**Result: reproduced.** Under hd-a and rate-b, pairs closer than 60 degrees are more correlated in ADn than in PoS in every state. The medians are 0.28 against 0.06 in waking, 0.15 against 0.02 in SWS and 0.23 against 0.06 in REM, with Mann-Whitney p = 1.9e-7, 4.6e-8 and 1.7e-6, each below the text's 1e-5. Under hd-a and rate-a, PoS has only 12 pairs and p is 0.03 to 0.045. The ADn pair counts are 328 to 480; 328 (hd-a, rate-a) and 374 (hd-b, rate-a) are within 345 plus or minus 34.5.

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P0 = hdpairs(rd, rate)
        P0 = P0[P0.kind.isin(["ADn-ADn", "PoS-PoS"])]
        pk = CELLS.set_index(["session", "unit"])["peak_" + rd]
        P0 = P0.assign(pk1=pk.loc[list(zip(P0.session, P0.u1))].values, pk2=pk.loc[list(zip(P0.session, P0.u2))].values)
        for which in ("all", "<60"):
            P1 = P0 if which == "all" else P0[P0["off_" + rd] < 60]
            for rmeas in ("geo_peak", "mean_rate_state", "mean_rate_wake"):
                for st in ("wake", "SWS", "REM", "pooled"):
                    sts = ("wake", "SWS", "REM") if st == "pooled" else (st,)
                    parts_ = []
                    for k in sts:
                        D = P1.assign(corr=P1["corr_" + k], state=k)
                        if rmeas == "geo_peak":  # peak rates of the HD reading's fit
                            rr = np.sqrt(D.pk1.astype(float) * D.pk2.astype(float))
                        elif rmeas == "mean_rate_state":  # mean rates in the state tested
                            rr = 0.5 * (D["rate_" + k + "1"].astype(float) + D["rate_" + k + "2"].astype(float))
                        else:  # waking mean rates (variant)
                            rr = 0.5 * (D.rate_wake1.astype(float) + D.rate_wake2.astype(float))
                        parts_.append(D.assign(rg=np.where(rr < 5, "lt5", np.where(rr < 10, "5to10", "gt10"))))
                    D = pd.concat(parts_)
                    D = D[(D.rg != "gt10") & D["corr"].notna()]
                    if D.kind.nunique() < 2 or D.rg.nunique() < 2:
                        continue
                    m = smf.ols("corr ~ C(kind) + C(rg)", data=D).fit()
                    a = sm.stats.anova_lm(m, typ=2)
                    rows.append(dict(hd=rd, rate=rate, pairs=which, rate_measure=rmeas, state=st, n=len(D),
                                     p_region=a.loc["C(kind)", "PR(>F)"], p_rategroup=a.loc["C(rg)", "PR(>F)"]))
S5 = pd.DataFrame(rows)
print(S5.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for rm, g in S5.groupby("rate_measure"):
    print(f"{rm}: readings {len(g)}; region p < 1e-7 in {int((g.p_region < 1e-7).sum())}; rate-group p > 0.05 in "
          f"{int((g.p_rategroup > 0.05).sum())}; both in {int(((g.p_region < 1e-7) & (g.p_rategroup > 0.05)).sum())}; "
          f"rate-group p < 0.05 where region p < 1e-7: {int(((g.p_region < 1e-7) & (g.p_rategroup < 0.05)).sum())}")
print("all:", len(S5), "readings; region p < 1e-7:", int((S5.p_region < 1e-7).sum()),
      "| rate-group p > 0.05:", int((S5.p_rategroup > 0.05).sum()), "| both:", int(((S5.p_region < 1e-7) & (S5.p_rategroup > 0.05)).sum()))

**Result: not reproduced.** The rate-group effect is significant wherever the region effect reaches p < 1e-7. Of 96 readings (HD, rate threshold, pairs, rate measure, state), the region effect reaches p < 1e-7 in 12, all among pairs closer than 60 degrees, and in each of them the rate-group effect has p from 4.8e-14 to 0.0013. No reading has region p < 1e-7 together with rate-group p > 0.05. Over all pairs the region effect is not found (smallest p 0.029). The rate group now uses the peak rate of the HD reading's own fit, and the mean rate of the state tested (waking rates as a variant).

In [ ]:
print("ensembles:", ENS.groupby("region").size().to_dict())
CO = COH[~COH.ref.str.endswith("_shuffled")]
T3B = CO.pivot_table(index=["region", "ref", "dist"], columns="state", values=["mean", "median"], aggfunc="median")
print("median over ensembles of the per-ensemble summary of coherence:")
print(T3B.round(3).to_string())
lo = CO.groupby(["ref", "dist", "state", "region"])[["mean", "median"]].min()
print("\nminimum over ensembles:")
print(lo.unstack(["state", "region"]).round(3).to_string())
print("ensembles with coherence above 0.05 (fraction), by reading:")
print(CO.assign(ok_mean=CO["mean"] > 0.05, ok_median=CO["median"] > 0.05).groupby(["ref", "dist", "state", "region"])[["ok_mean", "ok_median"]].mean().unstack(["state", "region"]).round(2).to_string())
SH = COH[COH.ref.str.endswith("_shuffled")]
print("\nnegative control: the same packets against the expected packet at decoded angles permuted across time bins;"
      " median over ensembles:")
print(SH.pivot_table(index=["region", "ref", "dist"], columns="state", values=["mean", "median"], aggfunc="median").round(3).to_string())
print("fraction of ensembles whose shuffled coherence exceeds 0.05 (mean summary):")
print(SH.assign(ok=SH["mean"] > 0.05).groupby(["ref", "dist", "region"]).ok.mean().unstack("region").round(2).to_string())

**Result: reproduced.** At the decoder's windows, the median over ensembles of the coherence exceeds 0.05 in both structures and all three states under every reading of distance, reference vectors and summary. The lowest is 0.082 (ADn, SWS, decoded-HD references, correlation distance, median over bins). Every ensemble passes with the mean summary. The shuffled control shows which readings can fail: packets compared with the expected packet at permuted decoded angles still score 0.25 to 0.61 with random references and 0.11 to 0.24 with actual-HD references and the mean summary (medians over ensembles), above 0.05, so those readings cannot fail. With decoded-HD references and the median summary the shuffled control is 0.000 in ADn and 0.000 to 0.122 in PoS, against 0.082 to 0.503 for the actual angles, so the result rests on that reading. Other windows were not run.

## Noise correlations and linear decoding: fig4b-noise-correlations, fig4c-noise-adn-vs-pos, fig4d-ole-near-bayesian, fig4e-ole-covariance

Noise correlation (Methods, Cross-correlation under the hypothesis of independent firing rate coding): spike-train minus intensity-function cross-correlation, 10 ms bins, 1/N sum of products, waking bins with head direction; readings (n-a) mean over -50 to +50 ms or the 0 lag, (n-b) raw (Hz^2) or divided by the product of mean rates.

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_ = hdpairs(rd, rate)
        for meas in ("noise_avg_raw", "noise_0_raw", "noise_avg_norm", "noise_0_norm"):
            for grp, c in (("ADn-ADn", P_[P_.kind == "ADn-ADn"]), ("pooled", P_)):
                o = c["off_" + rd]
                z = c[o <= 15][meas].dropna(); s60 = c[(o >= 45) & (o <= 75)][meas].dropna(); far = c[o > 90][meas].dropna()
                p0 = stats.wilcoxon(z).pvalue if len(z) > 5 else np.nan
                p6 = stats.wilcoxon(s60).pvalue if len(s60) > 5 else np.nan
                rows.append(dict(hd=rd, rate=rate, measure=meas, pairs=grp, n0=len(z), med0=z.median(), p0=p0,
                                 n60=len(s60), med60=s60.median(), p60=p6, mean_far=far.mean(),
                                 holds=(z.median() > 0) & (p0 < 0.05) & (s60.median() < 0) & (p6 < 0.05)))
F4B = pd.DataFrame(rows)
print(F4B.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
g = F4B.groupby(["hd", "rate", "measure"]).holds.all()
print("\nreadings where both ADn-ADn and pooled pairs hold:", g[g].index.tolist())

**Result: reproduced.** In every reading (HD, rate, lag and scale), the noise correlation is above 0 for pairs within 15 degrees and below 0 for pairs at 60 plus or minus 15 degrees, in ADn-ADn pairs and in pooled pairs. Every Wilcoxon p is below 1e-11. Under hd-a, rate-a and the -50 to +50 ms normalised reading, the ADn-ADn medians are 0.89 and -0.30, and beyond 90 degrees the mean is -0.02.

In [ ]:
rows = []
for rd in ("a", "b"):
    for rate in ("a", "b"):
        P_ = hdpairs(rd, rate)
        for meas in ("noise_avg_raw", "noise_0_raw", "noise_avg_norm", "noise_0_norm"):
            for grp, sel in (("0", P_["off_" + rd] <= 15), ("60", (P_["off_" + rd] >= 45) & (P_["off_" + rd] <= 75))):
                c = P_[sel]
                g_ = [c[c.kind == k][meas].dropna() for k in ("ADn-ADn", "ADn-PoS", "PoS-PoS")]
                if min(len(x) for x in g_) < 2:
                    rows.append(dict(hd=rd, rate=rate, measure=meas, group=grp, n=[len(x) for x in g_], p=np.nan))
                    continue
                kw = stats.kruskal(*g_)
                meds = [x.median() for x in g_]
                ext = (np.argmax(meds) == 0) if grp == "0" else (np.argmin(meds) == 0)
                rows.append(dict(hd=rd, rate=rate, measure=meas, group=grp, n=[len(x) for x in g_],
                                 med_adn=meds[0], med_mixed=meds[1], med_pos=meds[2], p=kw.pvalue, adn_most_extreme=ext))
F4C = pd.DataFrame(rows)
print(F4C.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("expected pairs at 0 / 60 deg: ADn-ADn 96 / 200, ADn-PoS 77 / 157, PoS-PoS 29 / 57")
ok = F4C.assign(ok=(F4C.p < 0.05) & F4C.adn_most_extreme.fillna(False).astype(bool)).groupby(["hd", "rate", "measure"]).ok.sum()
print("groups (of 2) passing per reading:", ok.to_dict())

**Result: reproduced.** Under hd-a and rate-b, with every lag and scale reading, the Kruskal-Wallis test across the three pair types gives p < 0.001 at 0 and at 60 degrees, with ADn-ADn pairs the most extreme. The raw 50 ms reading gives p = 8.5e-9 and 2.3e-13. At 0 degrees there are 101 ADn-ADn, 49 ADn-PoS and 19 PoS-PoS pairs, against 96, 77 and 29. At 60 degrees there are 217, 107 and 31, against 200, 157 and 57.

In [ ]:
# OLE (hd.ole_fit, hd.ole_errors; assumed estimator, see the claim's criterion):
# covariance OLE, Q = <r r^T> of the smoothed counts; signal OLE, adopted reading
# "sig", Q = <F F^T> of the tuning curves read at the head direction off the
# diagonal, with the observed variances on it (independent neurons, the usual
# signal-only OLE); variant "sig2", Q = <F F^T> throughout, whose decoder errs
# near or beyond chance (90 deg) in some ensembles. Ensembles of HD cells under
# (hd-a) and (hd-b), sessions with at least 6 in a structure.
W_ = hd.OLE_WINDOWS
E4 = ENS_ALL.copy()
for k in ("ole_cov", "ole_sig", "ole_sig2", "bayes"):
    E4[k + "_min"] = E4[[f"{k}_{w}" for w in W_]].min(axis=1)
    E4[k + "_argmin"] = E4[[f"{k}_{w}" for w in W_]].values.argmin(axis=1)
print(E4[["reading", "session", "region", "n_hd", "ole_cov_min", "ole_sig_min", "ole_sig2_min", "bayes_min"]].round(2).to_string(index=False))
print("windows (s):", [float(w) for w in W_])
print("signal OLE readings: sig, Q = <F F^T> off the diagonal with the observed variances on it (adopted); "
      "sig2, Q = <F F^T> throughout (variant)")
rows = []
for rd, er in E4.groupby("reading"):
    for sig in ("ole_sig", "ole_sig2"):
        for reg, e in er.groupby("region"):
            w = stats.wilcoxon(e.ole_cov_min, e[sig + "_min"])
            rows.append(dict(hd=rd, signal=sig, region=reg, n=len(e), cov=e.ole_cov_min.median(), signal_err=e[sig + "_min"].median(),
                             signal_max=e[sig + "_min"].max(), bayes=e.bayes_min.median(),
                             cov_better=int((e.ole_cov_min < e[sig + "_min"]).sum()), wilcoxon_p=w.pvalue,
                             cov_over_bayes=e.ole_cov_min.median() / e.bayes_min.median()))
F4D = pd.DataFrame(rows)
print(F4D.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for (rd, sig), f in F4D.groupby(["hd", "signal"]):
    first = bool((f.wilcoxon_p < 0.05).all() and (f["cov"] < f.signal_err).all())
    second = bool((f.cov_over_bayes <= 1.25).all())
    print(f"{rd}, {sig}: covariance OLE better with p < 0.05 in both structures {first}; within 25% of Bayes in both {second}")
for (rd, reg), e in E4.groupby(["reading", "region"]):
    print(f"{rd} {reg}: per-ensemble cov / Bayes median {(e.ole_cov_min / e.bayes_min).median():.3f}, range "
          f"{(e.ole_cov_min / e.bayes_min).min():.3f} to {(e.ole_cov_min / e.bayes_min).max():.3f}")
print("expected ensembles: 20 ADn, 8 PoS")


**Result: partially reproduced under (hd-b); not reproduced under (hd-a).** With the adopted signal OLE (observed variances on the diagonal of Q) and hd-a ensembles, the covariance OLE barely beats the signal OLE (median minimum error 12.98 against 13.22 degrees in ADn, better in 14 of 25 ensembles, Wilcoxon p = 0.16; PoS 20.96 against 20.96, p = 0.13, 5 ensembles), so the first part fails. With hd-b ensembles (27 ADn, 11 PoS) it holds in both structures: 13.06 against 13.81 degrees in ADn (17 of 27, p = 0.019) and 27.79 against 28.37 in PoS (9 of 11, p = 0.042). The covariance OLE errors are 1.26 (ADn) and 1.42 (PoS) times the Bayesian decoder's under hd-a, and 1.24 and 1.76 under hd-b, so the second part fails in PoS under both (ADn is within the margin under hd-b only). With the variant signal Q (tuning curves throughout), whose signal decoder errs by up to 103 degrees under both HD readings, beyond the 90 degrees of chance, the first part holds in ADn under both readings and in PoS under hd-b (p = 0.001) but not under hd-a (p = 0.0625, the floor for 5 pairs). So under (hd-b) the first part holds in both structures with either signal Q, and the criterion makes the claim partially reproduced; under (hd-a) it is not reproduced with either. The outcome is partially reproduced, the change proposed after round 2 and agreed in the third verification round. The gain in ADn is about 1% of the error, far from the paper's "approximately 50%", which fig4e-ole-covariance tests. The outcome rests on the estimator assumed here, as the paper defines none, and on the HD reading.


In [ ]:
rows = []
for rd, er in E4.groupby("reading"):
    for sig in ("ole_sig", "ole_sig2"):
        r = er.ole_cov_min / er[sig + "_min"]
        for reg in ("ADn", "PoS"):
            rr = r[er.region == reg]
            print(f"{rd} {sig} {reg}: n {len(rr)}, median ratio cov/signal {rr.median():.3f}, range {rr.min():.3f} to {rr.max():.3f}")
        u = stats.mannwhitneyu(r[er.region == "ADn"], r[er.region == "PoS"], alternative="two-sided")
        print(f"{rd} {sig} Mann-Whitney ADn against PoS: U {u.statistic:.1f}, p {u.pvalue:.3g}; ADn median lower: "
              f"{r[er.region == 'ADn'].median() < r[er.region == 'PoS'].median()}")
print("readings: median ratio 0.5 (0.45 to 0.55) or 0.67 (0.60 to 0.73); expected 20 ADn and 8 PoS ensembles")
print("ensembles:", E4.groupby(["reading", "region"]).size().to_dict())


**Result: not reproduced.** With the adopted signal OLE (observed variances on the diagonal), the median ratio of covariance to signal OLE error is 0.99 in ADn (range 0.91 to 1.04) and 0.98 in PoS under hd-a, and 0.99 and 0.99 under hd-b: the covariance OLE is about as good as the signal OLE, not 50% better, and ADn and PoS do not differ (Mann-Whitney p = 0.28 and 0.82). With the variant signal Q (tuning curves throughout), the ADn ratio is 0.25 (hd-a) or 0.25 (hd-b), outside both 0.45 to 0.55 and 0.60 to 0.73, as that signal decoder works poorly, and ADn against PoS gives p = 0.17 and 0.067. No reading gives either part. There are 25 ADn and 5 PoS ensembles under hd-a and 27 and 11 under hd-b, against 20 and 8. The outcome rests on the estimator assumed here.


## Bayesian decoding: decoding-tracks-hd, fig5b-sleep-tuning-similar, fig5b-adn-decoded-tuning

In [ ]:
print(ENS[["session", "region", "n_hd", "err_full", "err_cv"]].round(2).to_string(index=False))
for reg, e in ENS.groupby("region"):
    print(f"{reg}: n {len(e)}; median over ensembles of the median absolute error: whole-epoch tuning curves "
          f"{e.err_full.median():.2f} deg (max {e.err_full.max():.2f}), cross-validated {e.err_cv.median():.2f} deg (max {e.err_cv.max():.2f})")
print("threshold 30 deg; chance 90 deg; shifted head direction (data check) gives", piv.median().round(1).to_dict())
EB = ENS_ALL[ENS_ALL.reading == "hd-b"]
print("ensembles of hd-b cells (waking decoding only):")
print(EB[["session", "region", "n_hd", "err_full", "err_cv"]].round(2).to_string(index=False))
for reg, e in EB.groupby("region"):
    print(f"hd-b {reg}: n {len(e)}; median over ensembles of the median absolute error: whole-epoch tuning curves "
          f"{e.err_full.median():.2f} deg (max {e.err_full.max():.2f}), cross-validated {e.err_cv.median():.2f} deg (max {e.err_cv.max():.2f})")


**Result: reproduced.** The median over ensembles of the median absolute decoding error is 10.2 degrees in 25 ADn ensembles (maximum 13.1) and 14.4 degrees in 5 PoS ensembles (maximum 25.1) with whole-epoch tuning curves of hd-a cells. Cross-validated, it is 10.7 and 14.8. With hd-b ensembles (27 ADn, 11 PoS) it is 10.1 and 15.0, or 10.7 and 16.1 cross-validated. All are under 30 degrees, against 76 to 77 degrees with the head direction shifted by 10 s.


In [ ]:
D5 = DEC5B[DEC5B.hd_a.astype(bool)]
print(D5.groupby(["decoder", "target_region", "state"]).agg(n=("r_a", "size"), median_r_a=("r_a", "median"),
                                                              median_r_b=("r_b", "median")).round(3).to_string())
print("sessions:", D5.groupby("decoder").session.nunique().to_dict(), "| threshold median r >= 0.5")
print("by session, ADn-decoded PoS cells, median r_a per state:")
print(D5[D5.decoder == "ADn"].groupby(["session", "state"]).r_a.median().unstack().round(2).to_string())

**Result: reproduced.** The tuning curves against decoded HD correlate with the waking tuning curves with medians of 0.92 (REM), 0.90 (SWS) and 0.98 (waking) for 51 to 64 PoS HD cells decoded from ADn, and 0.87, 0.81 and 0.93 for 43 to 52 ADn HD cells decoded from PoS. Both bin readings give the same medians within 0.01. Every median is above 0.5.

In [ ]:
for col in ("r_a", "r_b"):
    for st in ("wake", "SWS", "REM"):
        A = D5[(D5.decoder == "ADn") & (D5.state == st)][col].dropna()
        B = D5[(D5.decoder == "PoS") & (D5.state == st)][col].dropna()
        if len(A) and len(B):
            u = stats.mannwhitneyu(A, B, alternative="two-sided")
            print(f"{col} {st}: PoS cells (ADn decoded) n {len(A)} median {A.median():.3f}; ADn cells (PoS decoded) n {len(B)} "
                  f"median {B.median():.3f}; Mann-Whitney p {u.pvalue:.3g}")
print("expected 86 PoS and 92 ADn HD cells; paper p: wake < 1e-10, SWS 0.018, REM 3e-6")

**Result: reproduced.** PoS cells decoded from ADn have higher correlations than ADn cells decoded from PoS in every state. Under reading (a), Mann-Whitney p = 7.0e-7 in waking, 5.0e-4 in SWS and 0.039 in REM; reading (b) gives 1.5e-7, 4.7e-4 and 0.042. The counts are 64 and 52 cells in waking, against 86 and 92.

## Fast synchrony: fig6a-fast-oscillation, fig6b-adn-synchrony

Cross-correlograms of ADn HD cell pairs in 0.5 ms bins to +-50 ms (readings o-a, o-b), z-scored against the jitter null. The mean and s.d. of the jittered cross-correlogram (one train jittered uniformly by +-10 ms) are taken as the cross-correlogram convolved with a 20 ms box and its Poisson s.d. These are the expectation of the paper's 100 jitters, not the 100 jitters themselves (deviation). The power is from a complex Morlet transform (o-c), averaged over lags within +-25 ms.

In [ ]:
FA = attach(FAST)
FA = FA[FA.hd_a1.astype(bool) & FA.hd_a2.astype(bool)]
rows = []
for shank in ("different shanks", "all pairs"):
    f = FA if shank == "all pairs" else FA[~FA.same_shank]
    for st, c in f.groupby("state"):
        pw = np.vstack(c.power.values).mean(0)
        sel = hd.FAST_FREQS > 80
        fpk = hd.FAST_FREQS[sel][np.argmax(pw[sel])]
        rows.append(dict(pairs=shank, state=st, n=len(c), peak_freq=fpk, in_117_176=117 <= fpk <= 176,
                         power_at_peak=pw[sel].max(), power_80=pw[hd.FAST_FREQS == 80][0], power_300=pw[-1]))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for i, shank in enumerate(("different shanks", "all pairs")):
    f = FA if shank == "all pairs" else FA[~FA.same_shank]
    for st, c in f.groupby("state"):
        ax[i].plot(hd.FAST_FREQS, np.vstack(c.power.values).mean(0), label=st)
    ax[i].set_title(shank); ax[i].set_xlabel("Hz")
ax[0].set_ylabel("mean power (z units^2)"); ax[0].legend(); plt.tight_layout(); plt.show()

**Result: reproduced.** For ADn HD pairs on different shanks (492 per state), the mean power of the z-scored cross-correlograms above 80 Hz peaks at 165 Hz in waking, 175 Hz in SWS and 165 Hz in REM, all within 117 to 176 Hz. With same-shank pairs included, the peak moves to the top of the frequency grid (285 to 300 Hz).

In [ ]:
rows = []
for amp in ("zpeak5", "z0"):
    for shank in ("different shanks", "all pairs"):
        f = FA if shank == "all pairs" else FA[~FA.same_shank]
        f = f.assign(grp=np.minimum((f.off_a // 30).astype(int), 5))
        for st in ("wake", "SWS", "REM", "pooled"):
            c = f if st == "pooled" else f[f.state == st]
            g_ = [c[c.grp == k][amp].values for k in range(6)]
            kw = stats.kruskal(*[x for x in g_ if len(x)])
            means = [x.mean() if len(x) else np.nan for x in g_]
            rows.append(dict(amplitude=amp, pairs=shank, state=st, n=len(c), p=kw.pvalue, mean_0_30=means[0],
                             mean_150_180=means[5], falls=means[0] > means[5],
                             monotonic=bool(np.all(np.diff(means) <= 0)), means=np.round(means, 2).tolist()))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("criterion: 'falls' is the mean of the first group above that of the last; 'monotonic' (not required) is shown beside it")
print("ADn HD pairs per state:", FA.groupby("state").size().to_dict(), "| expected 970")

**Result: reproduced.** With the peak z within 5 ms and pairs on different shanks, the three states pooled give Kruskal-Wallis p = 2.2e-24 across 30 degree offset groups. The mean falls from 11.6 (0 to 30 degrees) to 3.0 (150 to 180 degrees), the fall the criterion asks for. In each state p < 0.05 with the same fall (waking p = 1.4e-5, SWS 5.1e-14, REM 2.2e-6). Waking alone does not reach 1e-10. The fall is not monotonic: in every state the mean rises again in the 120 to 150 degree group (for example 2.94 to 3.65 pooled). The zero-lag z of all pairs is negative (same-shank pairs).

## ADn to PoS: fig7-adn-pos-connections, fig7c-adn-triggered-pos-lfp, fig7d-same-preferred-direction, fig7e-driven-pos-hd-information, fig7e-interneurons-no-difference

`hd.mono_test`: cross-correlograms of every ADn-PoS pair over the whole recording, 0.5 ms bins; baseline the cross-correlogram convolved with a 10 ms s.d. Gaussian, with (hollow) or without the central bin (m-b), or the jitter expectation (m-d, 20 ms box); two consecutive bins above the 99.9th Poisson percentile at lags of 1 to 10 ms, positive lags ADn to PoS. Sessions with both structures by the files' labels (criterion m-a, "in sessions with both structures"), including Mouse28-140314, which has no head direction (connections need none); the paper's three mice are a variant.

In [ ]:
def mono_table(MO_):
    MO_ = MO_.copy()
    key_a, key_p = list(zip(MO_.session, MO_.adn)), list(zip(MO_.session, MO_.pos))
    ka, kp = pd.MultiIndex.from_tuples(key_a), pd.MultiIndex.from_tuples(key_p)
    MO_["adn_hd"] = C_idx.hd_a.reindex(ka).fillna(False).values.astype(bool)
    MO_["pos_hd"] = C_idx.hd_a.reindex(kp).fillna(False).values.astype(bool)
    MO_["pos_cls"] = C_idx.cls.reindex(kp).fillna("und").values
    return MO_
MO_ALL = mono_table(MONO)
MO = MO_ALL[MO_ALL.subject.isin(BOTH)]
rows = []
for sample, M_ in (("files' labels (4 mice)", MO), ("paper's 3 mice", MO[MO.subject.isin(DUAL)]),
                   ("paper's 3 mice, sessions with head direction", MO[MO.subject.isin(DUAL) & MO.session.isin(CELLS.session)])):
    for base in ("c", "h", "j"):
        for pairs in ("all ADn x all PoS", "ADn HD x all PoS"):
            m = M_ if pairs.startswith("all") else M_[M_.adn_hd]
            py = M_[M_.adn_hd & (M_.pos_cls == "pyr")]
            a2p, p2a = m["a2p_" + base].sum(), m["p2a_" + base].sum()
            ya, yp = py["a2p_" + base].sum(), py["p2a_" + base].sum()
            b1 = stats.binomtest(int(ya), int(ya + yp), 0.5).pvalue if ya + yp else np.nan
            b2 = stats.binomtest(int(ya), len(py), yp / len(py)).pvalue if yp else np.nan
            rows.append(dict(sample=sample, baseline=base, pairs=pairs, n=len(m), frac_a2p=a2p / len(m), frac_p2a=p2a / len(m),
                             n_a2p=int(a2p), n_p2a=int(p2a), n_pyr_pairs=len(py), frac_pyr_a2p=ya / len(py),
                             frac_pyr_p2a=yp / len(py), n_pyr_a2p=int(ya), n_pyr_p2a=int(yp), binom_half=b1, binom_prop=b2))
F7 = pd.DataFrame(rows)
print(F7.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("expected: 6960 pairs, 4.45% ADn to PoS, 0.35% PoS to ADn; 3394 pyramidal pairs, 0.41% and 0.18%")
print("by mouse (baseline c, all pairs):", MO.groupby("subject")[["a2p_c", "p2a_c"]].mean().round(4).to_dict())
per = MO.groupby("session").agg(pairs=("adn", "size"), a2p=("a2p_c", "sum"), p2a=("p2a_c", "sum"))
per["n_adn_x_n_pos"] = SUM.set_index("session").reindex(per.index).eval("n_adn * n_pos")
print("pairs per session (baseline c):")
print(per.to_string())
print("most ADn x PoS pairs any session of the dandiset can give (sum of n_adn x n_pos over sessions):",
      int((SUM.n_adn * SUM.n_pos).sum()), "| in the paper's 3 mice:", int((SUM.n_adn * SUM.n_pos)[SUM.subject.isin(DUAL)].sum()),
      "| against 6960")

**Result: not reproduced.** The ADn to PoS fraction is within tolerance, but the pair count, the PoS to ADn fraction and the pyramidal fractions are not. Over all 4765 ADn by PoS pairs of the sessions with both structures, 4.11% are connected ADn to PoS (Methods' baseline; 4.01% with the jitter baseline) and 0.44% PoS to ADn, against 0.35% (0.315 to 0.385). 4765 is every ADn by PoS pair the dandiset holds (the sum of n_adn x n_pos over sessions), so 6960 cannot be reached; the paper's three mice give 4699 pairs (4.17% and 0.45%), and without Mouse28-140314, 4111 (4.57% and 0.51%). Among the 1583 pairs of ADn HD and PoS pyramidal cells, 1.07% are connected ADn to PoS and 0.44% PoS to ADn (17 and 7 pairs), against 0.41% and 0.18% over 3394. The direction holds against the PoS to ADn proportion (binomial p = 0.0009) and not against one half (17 of 24, p = 0.064).

In [ ]:
for sample, L in (("files' labels (4 mice)", LFP[LFP.subject.isin(BOTH)].copy()), ("paper's 3 mice", LFP[LFP.subject.isin(DUAL)].copy())):
    for ctl in ("jit", "rand"):
        L["ratio_" + ctl] = L.p_sta / L["p_" + ctl]
        w = stats.wilcoxon(L.p_sta, L["p_" + ctl])
        ses = L.groupby("session")["ratio_" + ctl].median()
        print(f"{sample}, control {ctl}: cells {len(L)}, sessions {L.session.nunique()}; median power ratio {L['ratio_' + ctl].median():.2f}; "
              f"cells above control {int((L['ratio_' + ctl] > 1).sum())}; sessions with median ratio > 1: {int((ses > 1).sum())} of {len(ses)}; Wilcoxon p {w.pvalue:.3g}")
L = LFP[LFP.subject.isin(BOTH)].copy()
print("spikes per cell in the window:", int(L.nspk.min()), "to", int(L.nspk.max()))
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
lag = (np.arange(len(L.sta_bp.iloc[0])) - len(L.sta_bp.iloc[0]) // 2) / 1.25
ax.plot(lag, np.vstack(L.sta_bp.values).mean(0)); ax.set_xlabel("lag (ms)"); ax.set_ylabel("80-250 Hz STA (a.u.)")
plt.tight_layout(); plt.show()
print("range of the plotted mean:", float(np.vstack(L.sta_bp.values).mean(0).min()), float(np.vstack(L.sta_bp.values).mean(0).max()))

**Result: reproduced.** Within 5 ms of ADn HD cell spikes, the 80 to 250 Hz PoS LFP has more power than at jittered spike times (median ratio 2.85, 126 of 175 cells, Wilcoxon p = 2.5e-12) and than at random times (3.17, p = 1.4e-13), in the sessions with both structures. The session median ratio exceeds 1 in 21 and 23 of 25 sessions. In the paper's three mice the ratios are 2.98 and 3.30 (18 and 20 of 20 sessions). One 14 minute window per session was read.

In [ ]:
ci = C_idx
for sample, M_ in (("files' labels (4 mice)", MO), ("paper's 3 mice", MO[MO.subject.isin(DUAL)])):
    for base in ("c", "h", "j"):
        c = M_[M_["a2p_" + base] & M_.adn_hd & M_.pos_hd]
        pa = ci.loc[list(zip(c.session, c.adn))].pref_a.values
        pp = ci.loc[list(zip(c.session, c.pos))].pref_a.values
        d = hd.circ_diff(pp.astype(float), pa.astype(float))
        if len(d) < 3:
            print(sample, base, "pairs", len(d)); continue
        R = np.abs(np.mean(np.exp(1j * d)))
        pr = hd.rayleigh_p(R, len(d))
        w = stats.wilcoxon(d)
        print(f"{sample}, baseline {base}: connected ADn HD to PoS HD pairs {len(d)}; median difference {np.degrees(np.median(d)):.1f} deg; "
              f"Rayleigh p {pr:.3g}; Wilcoxon against 0 p {w.pvalue:.3g}")
print("expected 12 pairs; paper Rayleigh p 2.4e-4, Wilcoxon p 0.68")

**Result: reproduced.** The 11 connected ADn HD to PoS HD pairs have unimodal differences of preferred direction (Rayleigh p = 3.0e-4) with no shift from 0 (median -8.8 degrees, Wilcoxon p = 0.41), under every baseline and in both samples of mice (Mouse20 adds no connected pair).

In [ ]:
def driven(base, cls, mice):
    pos = CELLS[(CELLS.region == "PoS") & CELLS.subject.isin(mice) & (CELLS.cls == cls)].copy()
    con = set(map(tuple, MO[MO["a2p_" + base] & MO.adn_hd][["session", "pos"]].values))
    pos["connected"] = [(s_, u_) in con for s_, u_ in zip(pos.session, pos.unit)]
    A, B = pos[pos.connected]["info"].dropna(), pos[~pos.connected]["info"].dropna()
    u = stats.mannwhitneyu(A, B, alternative="two-sided") if len(A) and len(B) else None
    print(f"{'4 mice' if len(mice) == 4 else '3 mice'}, baseline {base}, {cls}: connected n {len(A)} median {A.median():.3f}; "
          f"not connected n {len(B)} median {B.median():.3f}; difference {A.median() - B.median():.3f}; "
          f"Mann-Whitney p {u.pvalue if u else np.nan:.3g}")
for mice in (BOTH, DUAL):
    for base in ("c", "h", "j"):
        driven(base, "pyr", mice)
print("expected 11 connected and 170 unconnected pyramidal cells; paper p 0.011")

**Result: reproduced.** With the Methods' baseline (with or without the hollow), the 13 PoS pyramidal cells driven by an ADn HD cell carry more HD information than the 164 not connected (medians 1.08 against 0.58 bits per spike, Mann-Whitney p = 0.040), in both samples of mice. The result is fragile: with the jitter baseline variant (14 against 163), p = 0.075.

In [ ]:
for mice in (BOTH, DUAL):
    for base in ("c", "h", "j"):
        driven(base, "int", mice)
print("expected 13 connected and 79 unconnected interneurons; paper p 0.38")

**Result: reproduced.** The 8 driven PoS putative interneurons do not differ from the 59 not connected (medians 0.013 against 0.004 bits per spike, difference 0.009, Mann-Whitney p = 0.44; jitter baseline p = 0.58); in the paper's three mice, 8 against 51, p = 0.45. With 8 driven cells (13 in the paper) the test has little power.

## fig5d-adn-predicts-pos-reliably, fig5e-readout-windows, fig5e-adn-predicts-pos

These three claims rest on the GLM of Fig. 5d,e, which runs in `glm_peyrache_2015.ipynb`. fig5d-adn-predicts-pos-reliably and fig5e-adn-predicts-pos are tested there. fig5e-readout-windows is tested with its dense window grid in `glm_windows_peyrache_2015.ipynb`.

## Summary

| claim | outcome |
|---|---|
| recording-sample | partially reproduced |
| supptable1-per-animal | partially reproduced |
| hd-cell-fraction | not reproduced |
| hd-fraction-hd-shanks | reproduced |
| ensemble-sizes | partially reproduced |
| sleep-duration | reproduced |
| adn-pos-peak-rate | partially reproduced |
| adn-pos-hd-information | reproduced |
| supfig2a-unimodality | reproduced |
| fig2a-rates-preserved | reproduced |
| fig2b-pair-correlations-preserved | reproduced |
| fig2f-sws-tenfold-faster | partially reproduced |
| fig2e-decoded-speed | partially reproduced |
| fig2f-rem-similar-to-wake | reproduced |
| fig3a-packet-60-degrees | partially reproduced |
| fig3a-adn-stronger-than-pos | reproduced |
| supfig5-rate-independent | not reproduced |
| fig3b-packet-coherence | reproduced |
| fig4b-noise-correlations | reproduced |
| fig4c-noise-adn-vs-pos | reproduced |
| fig4d-ole-near-bayesian | partially reproduced (assumed estimator; hd-b ensembles) |
| fig4e-ole-covariance | not reproduced (assumed estimator) |
| decoding-tracks-hd | reproduced |
| fig5b-sleep-tuning-similar | reproduced |
| fig5b-adn-decoded-tuning | reproduced |
| fig5d-adn-predicts-pos-reliably | reproduced (glm_peyrache_2015.ipynb) |
| fig5e-readout-windows | partially reproduced (glm_windows_peyrache_2015.ipynb) |
| fig5e-adn-predicts-pos | reproduced (glm_peyrache_2015.ipynb) |
| fig6a-fast-oscillation | reproduced (assumed jitter null) |
| fig6b-adn-synchrony | reproduced (assumed jitter null) |
| fig7-adn-pos-connections | not reproduced |
| fig7c-adn-triggered-pos-lfp | reproduced (assumed window and measure) |
| fig7d-same-preferred-direction | reproduced |
| fig7e-driven-pos-hd-information | reproduced |
| fig7e-interneurons-no-difference | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.**
- Every session with head direction is used. The paper counted the ADn cells of the three dual mice from a subset of sessions, which the files do not mark. A depth-based reading of the Methods' session rule is shown as a variant for the counts.
- Units are assigned to structures by the files' labels, so Mouse20's 8 PoS units count as PoS; the paper lists no PoS probe in that mouse.
- The GLM (glm_peyrache_2015.ipynb) uses HD cells as assembly and targets, 7 readout windows and at most 20 minutes of each state; the readout windows of fig5e-readout-windows (glm_windows_peyrache_2015.ipynb) use 26 windows, 23 of them at 20 per decade from 20 to 251 ms, for PoS targets only.
- The fast-synchrony null is the expectation of the jittered cross-correlogram, not 100 jitters.
- The OLE is the standard Salinas and Abbott estimator; the signal OLE takes Q from the tuning curves off the diagonal and the observed variances on it (Q from the tuning curves throughout as a variant). Ensembles are decoded under both HD readings for fig4d,e and decoding-tracks-hd, and under (hd-a) for the other decoding claims.
- The spike-triggered LFP uses one 14 minute window per session.
- Two sessions without the paper's sleep scoring enter the waking analyses only, with the open-field epoch as wake.

**What was found about the dataset.**
- Mouse28-140314 and both Mouse32 sessions hold no head direction, so Mouse32's cells cannot be classified. Mouse32-140820 also has no epochs or sleep states.
- Mouse25-140131 and Mouse25-140204 lack the paper's sleep scoring.
- Mouse20 has 8 units labelled post-subiculum in 3 sessions, on electrode groups labelled 'postsub' of a device the files call the post-subicular probe, though the paper lists no PoS probe in that mouse.
- In Mouse17-130130, spikes continue 1101 s past the last sub-recording.
- The wake interval of Mouse12-120808 spans 130 by 97 cm of LED positions, against the 53 by 46 cm arena.
- Recorded unit counts match Supplementary Table 1 in Mouse12, Mouse17 and Mouse32 and in PoS; Mouse20 has 48 thalamic units against the table's 49.
- The file's own HD classification agrees with reading (hd-a) for 1007 of 1064 classifiable cells.
- The scored wake interval coincides with the open-field epochs (overlap 0.9998 to 1 both ways, 37 sessions).